# Генерація тексту

У цій роботі ми використаємо мовну модель для генерації тексту.

У класичної мовної моделі є два взаємопов'язані визначення:

1. Оцінити ймовірність вхідного тексту.
2. Видати ймовірнісний розподіл наступного слова для даного префіксу.

Для генерації тексту нам ідеально підходить друге визначення.

## Початок роботи

Будь ласка, заповніть поля `EMAIL`, `NAME` та `GROUP` нижче:

In [64]:
import lpnlp

lab = lpnlp.start(
    email="sviatoslav.humeniuk.shi.2023@lpnu.ua",
    lab="text_generation"
)

Удачі!


### Завантаження моделі

Тренування мовної моделі з нуля займає багато часу: від кількох годин або днів для маленьких та середніх моделей й аж до кількох місяців чи навіть років для великих. Звичайно, за рахунок розпаралелювання тренування на багатьох GPU, реальний час рідко буває більше місяця-двох.

Для цієї роботи (як і в реальному житті), ми візьмемо претреновану модель. Хтось натренував її за нас. В данному випадку, візьмемо GPT-2 від компанії OpenAI.

Ця модель, як і безліч інших, зберігається на [HuggingFace Models](https://huggingface.co/models). Завантажити та працювати з нею зручно через бібліотеку [HuggingFace Transformers](https://github.com/huggingface/transformers/)

In [65]:
import transformers
import torch

In [66]:
model = transformers.AutoModelForCausalLM.from_pretrained("gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

## Токенізація

Кожна модель має словник токенів, з яким вона тренувалася, та правила токенізації (розбиття тексту на токени). Обов'язково слід використовувати той самий словник та метод.

Бібліотека transformers вміє робити це для кожної підтримуваної моделі.

In [67]:
tokenizer = transformers.AutoTokenizer.from_pretrained("gpt2")

Перевіримо токенізатор:

In [68]:
input_text = "Hello, LP NLP!"
token_ids = tokenizer.encode(input_text)
token_ids

[15496, 11, 18470, 399, 19930, 0]

In [69]:
tokenizer.convert_ids_to_tokens(token_ids)

['Hello', ',', 'ĠLP', 'ĠN', 'LP', '!']

Рідкісні слова будуть розбиті на підслова:

In [70]:
tokenizer.convert_ids_to_tokens(tokenizer.encode("My name is Sviatoslav Humeniuk."))

['My', 'Ġname', 'Ġis', 'ĠSv', 'iat', 'oslav', 'ĠHum', 'eni', 'uk', '.']

## Перевірка моделі - один крок ітерації

Ми генеруватимемо текст в циклі токен за токеном, зліва направо. Але для початку розберімо, як виглядає один крок такого циклу.


На кожному кроці на вхід моделі подаємо ті токени, які вже згенеровано.

На першому кроці у нас ще нічого не згенеровано. Тому починаємо зі токену `<BOS>` ("begin of sentence"). В різних моделях він виглядає по-різному. Подивимося, що в GPT-2:


In [71]:
tokenizer.bos_token

'<|endoftext|>'

In [72]:
tokenizer.bos_token_id

50256

In [73]:
input = torch.LongTensor([[tokenizer.bos_token_id]])
input

tensor([[50256]])

In [74]:
output = model(input, return_dict=True)
output.logits.shape

torch.Size([1, 1, 50257])

Чому маємо саме таку розмірність?

* Перша одиниця - розмір батча, тобто кількість вхідних речень.
* Друга одиниця - це довжина вхідної послідовності. На першому кроці ми подали лише один токен (bos_token)
* 50257 - це розмір словника

In [75]:
tokenizer.vocab_size

50257

Поки що модель повернула не ймовірності, а просто якісь числа. Їм треба нормалізувати функцією softmax:

In [76]:
probs = torch.softmax(output.logits, dim=-1)

In [77]:
probs.shape

torch.Size([1, 1, 50257])

In [78]:
probs.sum()

tensor(1.0000, grad_fn=<SumBackward0>)

Ймовірності можуть ставати дуже маленькими та причиняти проблеми у зв'язку з обмеженою точністю float чисел. Тому прийнято працювати з логарифмами ймовірностей.

In [79]:
log_probs = torch.log_softmax(output.logits, dim=-1)
log_probs.exp().sum()

tensor(1.0000, grad_fn=<SumBackward0>)

Кожному слову в словнику відповідає своя ймовірність бути побаченим
після заданого префікса. Префіксом у нас поки що був лише одни `bos_token`.


In [80]:
for word in ("I", "red", "Why"):
    index = tokenizer.convert_tokens_to_ids(word)
    prob = log_probs[0, 0, index].exp()
    print(f"P({word}) = {prob}")

P(I) = 0.018320836126804352
P(red) = 0.00010652336641214788
P(Why) = 0.0008686658693477511


Тепер ми маємо ймовірностний розподіл по словнику. Можемо обрати слово, яке вважатимемо згенерованим. Тут можливі кілька стратегій, які ми розглянемо в наступних розділах.

## Greedy decoding

Найпростіший (але й не дуже цікавий) спосіб - це завжди обирати токен з найбільшою ймовірністю:

In [81]:
next_token_id = log_probs.argmax()
next_token_id

tensor(198)

In [82]:
probs[-1, -1, 198]

tensor(0.0623, grad_fn=<SelectBackward0>)

In [83]:
tokenizer.convert_ids_to_tokens([next_token_id])

['Ċ']

Зберемо код докупи та додамо цикл. В циклі ми продовжуватимемо генерувати текст токен за токеном, поки не настане одна з двох умов:
1. Модель видала спецальний токен `eos_token` (end of sentence)
2. Довжина згенерованого тексту перевищила певний поріг `max_len`

У хорошої моделі в більшості випадків має спрацьовувати перша умова зупинки. Проте іноді модель може впасти в безкінчений цикл. Щоб цьому запобігти, маємо другу умову.

In [84]:
@torch.no_grad()
def greedy_decode(model, tokenizer, max_len=50):
    start_index = tokenizer.bos_token_id
    result = [start_index]

    while len(result) < max_len:
        input = torch.LongTensor([result])
        output = model(input, return_dict=True)
        probs = torch.softmax(output.logits[0, -1], dim=-1)

        token_index = probs.argmax()

        if token_index == tokenizer.eos_token_id:
            break

        result.append(token_index.item())

    return result

generated_token_ids = greedy_decode(model, tokenizer)

Маємо список згенерованих індексів токенів, який починається ось так:

In [85]:
generated_token_ids[:10]

[50256, 198, 464, 717, 640, 314, 2497, 262, 649, 2196]

Перетворимо їх в текст:

In [86]:
tokens = tokenizer.convert_ids_to_tokens(generated_token_ids)
tokens[:10]

['<|endoftext|>',
 'Ċ',
 'The',
 'Ġfirst',
 'Ġtime',
 'ĠI',
 'Ġsaw',
 'Ġthe',
 'Ġnew',
 'Ġversion']

In [87]:
generated_text = "".join(tokens)
generated_text

'<|endoftext|>ĊTheĠfirstĠtimeĠIĠsawĠtheĠnewĠversionĠofĠtheĠgame,ĠIĠwasĠsoĠexcited.ĠIĠwasĠsoĠexcitedĠtoĠseeĠtheĠnewĠversionĠofĠtheĠgame,ĠIĠwasĠsoĠexcitedĠtoĠseeĠtheĠnewĠversionĠofĠtheĠgame,ĠIĠwasĠsoĠexcitedĠto'

In [88]:
lab.checkpoint("greedy decode", generated_text)

Відповідь правильна ✅



'<|endoftext|>ĊTheĠfirstĠtimeĠIĠsawĠtheĠnewĠversionĠofĠtheĠgame,ĠIĠwasĠsoĠexcited.ĠIĠwasĠsoĠexcitedĠtoĠseeĠtheĠnewĠversionĠofĠtheĠgame,ĠIĠwasĠsoĠexcitedĠtoĠseeĠtheĠnewĠversionĠofĠtheĠgame,ĠIĠwasĠsoĠexcitedĠto'

### Примітка: Byte-pair encoding (BPE)

Наша модель використовує subword токенізацію, а саме byte-pair encoding (BPE). В сучасному NLP це найрозповсюдженіший спосіб токенізації. Детально можете подивитися в [цьому відео](https://www.youtube.com/watch?v=tOMjTCO0htA).

Для наших цілей зараз важливо, що BPE заміняє пробіли на спеціальні Unicode-символи "Ġ". Серед інших моделей широко поширений варіант "▁" (зверніть увагу, це не звичайний символ підкреслення "_"). Щоб отримати чистий текст, треба виконати наступну заміну:

In [89]:
def bpe_decode(s):
    result = s.replace("Ġ", " ")
    result = result.replace("Ċ", "\n")
    return result

bpe_decode(generated_text)

'<|endoftext|>\nThe first time I saw the new version of the game, I was so excited. I was so excited to see the new version of the game, I was so excited to see the new version of the game, I was so excited to'

Але краще довіритися токенізатору й зробити цю роботу за нас:

In [90]:
tokenizer.convert_tokens_to_string(tokens)

'<|endoftext|>\nThe first time I saw the new version of the game, I was so excited. I was so excited to see the new version of the game, I was so excited to see the new version of the game, I was so excited to'

## Generic decoding function

Обирати слово з найбільшою ймовірністю -- не найкращий варіант для генерації тексту хоча б тому, що він завжди детерміновано призводить до однієї послідовності. Нижче ми подивимося на цікавіші альтернативи.

Цикл генерації залишиться той самий, що і в `greedy_decode()`. Відрізнятися буде лише один рядок -- той, в якому ми приймали рішення, яке слово обрати. Для зручності, винесемо цей рядок в окрему функцію. Ця функція прийматиме на вхід ймовірностний розподіл по словнику і повертає обраний токен.

In [91]:
def greedy_choice(probs):
    return probs.argmax()

Функцію генерації також трохи переробимо.

По-перше, додамо параметр `sample_fn` - це має бути функція, яка обирає слово з ймовірностного розподілу, наприклад, `greedy_choice`.

По-друге, для зручності виконуватимемо BPE декодинг у середині функції генерації.

In [92]:
@torch.no_grad()
def generate(model, tokenizer, sample_fn, max_len=50, bpe_decode=True):
    start_index = tokenizer.bos_token_id
    result = [start_index]

    while len(result) < max_len:

        input = torch.LongTensor([result])
        output = model(input, return_dict=True)
        probs = torch.softmax(output.logits[0, -1], dim=-1)

        token_index = sample_fn(probs)

        if token_index == tokenizer.eos_token_id:
            break

        result.append(token_index.item())

    if bpe_decode:
        tokens = tokenizer.convert_ids_to_tokens(result)
        result = tokenizer.convert_tokens_to_string(tokens)

    return result

generate(model, tokenizer, greedy_choice)

'<|endoftext|>\nThe first time I saw the new version of the game, I was so excited. I was so excited to see the new version of the game, I was so excited to see the new version of the game, I was so excited to'

## Simple sampling

Перший альтернатива - це sampling. Тепер ми обираємо наступний токен випадково, але пропорційно до ймовірностей.

In [93]:
def simple_sample(probs):
    return torch.multinomial(probs, num_samples=1)[0]

for i in range(1, 6):
    result = generate(model, tokenizer, simple_sample)
    print(f"#{i}: {result}")
    print()

#1: <|endoftext|>lots of anarcho-primitivism, but there are always exceptions.


Copro und Parmenides claims to insist on those particular communal policies which he defends, but messes up the real analysis by claiming that "every Christian

#2: <|endoftext|>garrettdynamix@gmail.com

WWW.garrettdynamix

#3: <|endoftext|>else if self . reports . string_messages_at == 0 &&

you. is_self (@hookitGifs) ||

hover[ 1 ]. behaviour . " Contribute " &&

self . reports .

#4: <|endoftext|>Hagerstown Heights, Ill. - The Spartanburg River and Hagerstown Park Council have requested the control and long term reservations for your resource allocation likely to generate and provide a great new local business environment for the housing community.

"

#5: <|endoftext|>Life isn't mix without passion and times are hard together. But if this is anything like the community cheering for Victor Oladipo, it seems to offer invalids an excuse to send umbrage: they may not love themselves.




## Sampling with temperature

Ми також можемо впливати на генерацію параметром температури softmax.

Більші значення температури призводять до того, що різниця між ймовірностями токенів зменшується, тобто розподіл стає більш рівномірним. На практиці це означає, що менш ймовірні варіанти обиратимуться частіше і згенерований текст може бути цікавішим. Однак якщо продовжувати піднімати температуру, то текст спочатку втратить зв'язність, далі почнуть розпадатися слова та граматичність.

Менші значення температури змінюють розподіл таким чином, що основна ймовірнісна маса припадає на невелику кількість топових токенів. При температурі 0 вся ймовірність дістанеться одному токену й семплінг перетвориться на greedy decoding.

Згенеруємо тексти з різною температурою:

In [94]:
for temperature in (0.1, 0.3, 0.5, 0.8, 1.0, 1.25, 1.5, 2.0, 3.0, 5.0):

    def sample_with_temp(probs):
        updated_probs = torch.softmax(torch.log(probs) / temperature, dim=-1)
        return simple_sample(updated_probs)

    print(f"Sampling with temperature={temperature}")
    result = generate(model, tokenizer, sample_with_temp, max_len=15)
    print(result)
    print()

Sampling with temperature=0.1
<|endoftext|>
The first time I saw the new version of the game, I

Sampling with temperature=0.3
<|endoftext|>
The first time I saw a video of a man in a wheelchair

Sampling with temperature=0.5
<|endoftext|>At the end of the day, the game is a game of words

Sampling with temperature=0.8
<|endoftext|>Harmony

The Little Kingdom is a romance novel set in

Sampling with temperature=1.0
<|endoftext|>Dronahan @ PleaseCryin United Player

Dear Dron

Sampling with temperature=1.25
<|endoftext|>60 14 Q A Buckingham Tower offered a test background story mostly about


Sampling with temperature=1.5
<|endoftext|>Roger: Love Fuller all schizophrenia disorder? Cohend Strislango Reese

Sampling with temperature=2.0
<|endoftext|>violitions study heating upgrade Tamaterasu Lessons Poly forks adaptations Chapter691 cut

Sampling with temperature=3.0
<|endoftext|>work ignitedures NYT standby Pag Like Jasper millionairePo project stun Courts posed

Sampling with temperatu

In [95]:
lab.checkpoint("softmax temperature", 0.8)

Відповідь правильна ✅
Окей, добре


0.8

## Top-k sampling

In [96]:
def top_k_sampling(probs, k):
    top_k = probs.topk(k)
    index = torch.multinomial(top_k.values, num_samples=1)[0]
    return top_k.indices[index]

In [97]:
k = 15

sample_fn = lambda probs: top_k_sampling(probs, k=k)
for i in range(1, 6):
    result = generate(model, tokenizer, sample_fn, max_len=20)
    print(f"#{i}: {result}")

#1: <|endoftext|>.

"When the time comes for the next president to take the oath of office,
#2: <|endoftext|>"I think it's the same with all the different types of vehicles, but it's a
#3: <|endoftext|>
The United Nations is expected to issue new guidelines in June that aim to end violence after more
#4: <|endoftext|>
A new law will allow local municipalities to make decisions on the city's parking fees and fees
#5: <|endoftext|>
The US has been a key player in the war against Isis militants who seized vast swaths


## Nucleus (top-p) sampling

In [98]:
def nucleus_sampling(probs, max_p):
    sorted_probs = probs.sort(descending=True)
    cummulative_prob = 0.0
    sample_indices = []
    sample_probs = []
    
    for i in range(0, len(sorted_probs.values)):
        p = sorted_probs.values[i]
        cummulative_prob += p
        sample_probs.append(p)
        sample_indices.append(sorted_probs.indices[i])
        
        if cummulative_prob >= max_p:
            break

    index = torch.multinomial(torch.tensor(sample_probs), num_samples=1)[0]
    return sample_indices[index]

In [99]:
for max_p in (0.0, 0.1, 0.3, 0.5, 0.6, 0.8, 1.0):
    sample_fn = lambda probs: nucleus_sampling(probs, max_p=max_p)
    result = generate(model, tokenizer, sample_fn, max_len=20)
    print(f"#{max_p}: {result}")

#0.0: <|endoftext|>
The first time I saw the new version of the game, I was so excited. I
#0.1: <|endoftext|>The House of Representatives has passed a bill that would allow the federal government to deny health insurance to
#0.3: <|endoftext|>.

The top 10 states for transgender people are:

.

.

#0.5: <|endoftext|>Please enable Javascript to watch this video

CHICAGO -- A group of students and
#0.6: <|endoftext|>We are committed to providing excellent service to our customers. We strive to provide our customers with the
#0.8: <|endoftext|>A flood of humanity takes place in our planet's natural systems of weather and temperature, as it
#1.0: <|endoftext|>Thank you some people. First check the page. Your email address will go into the information system


## Start from prompt

До цього моменту ми генерували текст з нуля. Однак значно кориснішим є задача генерації тексту від певного префікса або "підказки" - в англійській мові це називається "prompt".

Prompt дозволить нам контролювати тематику згенерованого тексту або, як ми побачимо на іншій лекції, допоможе моделі виконувати різноманітні завдання.

In [100]:
@torch.no_grad()
def generate(model, tokenizer, sample_fn, prompt, max_len=50, bpe_decode=True):

    result = tokenizer.encode(prompt)

    while len(result) < max_len:
        input = torch.LongTensor([result])
        output = model(input)
        log_probs = torch.log_softmax(output.logits[0, -1], dim=-1)

        token_index = sample_fn(log_probs.exp())

        if token_index == tokenizer.eos_token_id:
            break

        result.append(token_index.item())

    if bpe_decode:
        tokens = tokenizer.convert_ids_to_tokens(result)
        result = tokenizer.convert_tokens_to_string(tokens)

    return result

In [101]:
k = 3

sample_fn = lambda probs: top_k_sampling(probs, k=k)
for i in range(1, 5):
    result = generate(model, tokenizer, sample_fn, prompt="Philosophy is")
    print(f"#{i}: {result}")

#1: Philosophy is not the same as philosophy.

The first thing you should do is to understand that philosophy has no real meaning, and that philosophy is not the same as philosophy. Philosophy is not a science, but a system of beliefs,
#2: Philosophy is not a philosophy of the world, it is a philosophy of the human mind. It is not a philosophy of the world, but a philosophy of the mind. It is not a philosophy of the mind, but an understanding of the
#3: Philosophy is not a matter of being able to see through the veil of ignorance. It is not a matter of knowing the world. It is not a matter of knowing the world. The only way to know the world is to know the world
#4: Philosophy is a philosophy that is based on the belief that all life is a product of the creative process.

It is not an ideology, but rather a way of thinking that is based on the belief that all life is created by the


# Zero-shot класифікація

Ми будемо досліджувати цю тему на окремій лекції. Але спробуємо застосувати мовну модель для задачі класифікації вже зараз. Хоча наша модель надто маленька й слабка для серйозного використання в zero-shot.


In [102]:
generate(model, tokenizer, greedy_choice, prompt="This film is awful! Actor play is terrible. The plot is dull. I would rate it as a").splitlines()[0]

'This film is awful! Actor play is terrible. The plot is dull. I would rate it as a B+.'

In [103]:
generate(model, tokenizer, greedy_choice, prompt="This film is amazing! The time flies by when you watch it. Definitely recommend! On the scale of 1 to 5, I would rate it as a").splitlines()[0]

'This film is amazing! The time flies by when you watch it. Definitely recommend! On the scale of 1 to 5, I would rate it as a 5.'

In [104]:
lab.answer("Готово!")

Відповідь правильна ✅
Сподіваюсь, ця лаба була не надто важкою :)


'Готово!'

# Real world

У цій роботі ми імплементували кілька методів декодінгу. Але, звичайно, все вже зроблено за нас.

Подивіться на параметри функції [generate()](https://huggingface.co/docs/transformers/v4.21.1/en/main_classes/text_generation#transformers.generation_utils.GenerationMixin.generate) з бібліотеки transformers. Багато з них мають виглядати знайомими.

Повний приклад використання:

In [105]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

prompt = "Understanding the way GPT works"

input = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**input, do_sample=True, top_p=0.8, max_length=50)

tokenizer.batch_decode(outputs, skip_special_tokens=True)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

['Understanding the way GPT works:\n\nGPT is a global programmable programming language, and it has several benefits for each application: it allows for flexible and fast execution of operations on a large number of operating systems. GPT is also a']

# The End


In [106]:
lab.answer("Готово!")

Відповідь правильна ✅
Сподіваюсь, ця лаба була не надто важкою :)


'Готово!'